<a href="https://colab.research.google.com/github/zaamaulan/chatapp-hono/blob/main/GPT-SoVITS-colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# GPT-SoVITS Setup and Execution
This section handles the environment setup and launches the web interface.

In [1]:
import os
%cd /content
if not os.path.exists('GPT-SoVITS'):
    !git clone https://github.com/RVC-Boss/GPT-SoVITS
%cd /content/GPT-SoVITS

/content
/content/GPT-SoVITS


### Install Dependencies
We need to install system libraries for audio processing and the required Python packages.

In [3]:
!apt-get update && apt-get install -y ffmpeg opencc python3-dev build-essential
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install pyyaml ffmpeg-python opencc-python-reimplemented
# Remove opencc from requirements to avoid compilation issues if already installed
!sed -i '/^opencc$/d' requirements.txt
!pip install -r requirements.txt

Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:5 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:8 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading packag

### Download Pretrained Models
GPT-SoVITS requires base models. We will download them into the correct directory structure.

In [39]:
import os
from huggingface_hub import snapshot_download
%cd /content/GPT-SoVITS

# Ensure clean directories
base_path = "GPT_SoVITS/pretrained_models"
os.makedirs(base_path, exist_ok=True)

print("Downloading base models and tokenizers...")

# 1. Download Chinese RoBERTa (Tokenizer + Config + Weights)
snapshot_download(
    repo_id="hfl/chinese-roberta-wwm-ext-large",
    local_dir=f"{base_path}/chinese-roberta-wwm-ext-large",
    allow_patterns=["*.bin", "*.json", "*.txt"]
)

# 2. Download Chinese HuBERT base
snapshot_download(
    repo_id="TencentGameMate/chinese-hubert-base",
    local_dir=f"{base_path}/chinese-hubert-base"
)

# 3. Download GPT-SoVITS specific base models (V2Pro, V3, etc.)
# We use the official repository for these weights
snapshot_download(
    repo_id="lj1995/GPT-SoVITS",
    local_dir=base_path,
    allow_patterns=["v2Pro/*", "s1v3.ckpt", "s2Gv2.pth", "s2Gv1.pth", "s2D*.pth", "chinese-hubert-base/config.json"]
)

print("All pretrained models have been successfully downloaded using snapshot_download.")

/content/GPT-SoVITS


Fetching 7 files:   0%|          | 0/7 [00:00<?, ?it/s]

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

Fetching 7 files:   0%|          | 0/7 [00:00<?, ?it/s]

config.json: 0.00B [00:00, ?B/s]

v2Pro/s2Gv2ProPlus.pth:   0%|          | 0.00/200M [00:00<?, ?B/s]

v2Pro/s2Dv2ProPlus.pth:   0%|          | 0.00/126M [00:00<?, ?B/s]

s2D488k.pth:   0%|          | 0.00/93.5M [00:00<?, ?B/s]

All pretrained models have been successfully downloaded using snapshot_download.


In [42]:
import os

def check_models(path):
    print(f"Checking files in {path}:\n")
    for root, dirs, files in os.walk(path):
        for file in files:
            if file.endswith(('.pth', '.ckpt', '.bin')):
                full_path = os.path.join(root, file)
                size = os.path.getsize(full_path)
                # Files under a few KB are likely HTML error pages
                status = "OK" if size > 1024 * 1024 else "CORRUPT/SMALL"
                print(f"{status:15} | {size/1024/1024:8.2f} MB | {full_path}")
                if status == "CORRUPT/SMALL":
                    with open(full_path, 'r', errors='ignore') as f:
                        print(f"   Preview: {f.read(100)}...")

check_models('/content/GPT-SoVITS/GPT_SoVITS/pretrained_models')

Checking files in /content/GPT-SoVITS/GPT_SoVITS/pretrained_models:

OK              |    89.20 MB | /content/GPT-SoVITS/GPT_SoVITS/pretrained_models/s2D488k.pth
OK              |   148.09 MB | /content/GPT-SoVITS/GPT_SoVITS/pretrained_models/s1v3.ckpt
OK              |   190.85 MB | /content/GPT-SoVITS/GPT_SoVITS/pretrained_models/v2Pro/s2Gv2ProPlus.pth
OK              |   120.57 MB | /content/GPT-SoVITS/GPT_SoVITS/pretrained_models/v2Pro/s2Dv2Pro.pth
OK              |   120.58 MB | /content/GPT-SoVITS/GPT_SoVITS/pretrained_models/v2Pro/s2Dv2ProPlus.pth
OK              |   154.78 MB | /content/GPT-SoVITS/GPT_SoVITS/pretrained_models/v2Pro/s2Gv2Pro.pth
CORRUPT/SMALL   |     0.00 MB | /content/GPT-SoVITS/GPT_SoVITS/pretrained_models/sv/pretrained_eres2netv2w24s4ep4.ckpt
   Preview: {}...
OK              |   360.06 MB | /content/GPT-SoVITS/GPT_SoVITS/pretrained_models/chinese-hubert-base/pytorch_model.bin
OK              |  1245.96 MB | /content/GPT-SoVITS/GPT_SoVITS/pretrained_models/ch

In [43]:
from huggingface_hub import hf_hub_download
import os

# Target path for the speaker verification model
target_dir = "/content/GPT-SoVITS/GPT_SoVITS/pretrained_models/sv"
os.makedirs(target_dir, exist_ok=True)

print("Downloading corrected Speaker Verification model...")
hf_hub_download(
    repo_id="lj1995/GPT-SoVITS",
    filename="sv/pretrained_eres2netv2w24s4ep4.ckpt",
    local_dir="/content/GPT-SoVITS/GPT_SoVITS/pretrained_models"
)

print("Download complete. You can now restart the Web UI cell.")

sv/pretrained_eres2netv2w24s4ep4.ckpt:   0%|          | 0.00/108M [00:00<?, ?B/s]

Download complete. You can now restart the Web UI cell.


In [45]:
import os

# Create the missing directory for fast-langdetect
lang_detect_path = "/content/GPT-SoVITS/GPT_SoVITS/pretrained_models/fast_langdetect"
os.makedirs(lang_detect_path, exist_ok=True)

print(f"Created directory: {lang_detect_path}")
print("You can now try running the Web UI cell again.")

Created directory: /content/GPT-SoVITS/GPT_SoVITS/pretrained_models/fast_langdetect
You can now try running the Web UI cell again.


### Run the Web UI
Execute the following cell to start the Gradio interface. Use the public URL provided in the output to access the UI.

In [47]:
import os

# Set variabel lingkungan agar fast-langdetect tahu folder penyimpanannya
os.environ["is_share"] = "True"
os.environ["FAST_LANGDETECT_CACHE_DIR"] = "/content/GPT-SoVITS/GPT_SoVITS/pretrained_models/fast_langdetect"

%cd /content/GPT-SoVITS
!python webui.py

/content/GPT-SoVITS
Running on local URL:  http://0.0.0.0:9874
Running on public URL: https://6f6194631eeb515209.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from Terminal to deploy to Spaces (https://huggingface.co/spaces)
"/usr/bin/python3" -s GPT_SoVITS/inference_webui.py "Auto"
/content/GPT-SoVITS/GPT_SoVITS/inference_webui.py:1175: SyntaxWarning: invalid escape sequence '\d'
  parts = re.split("(\d+)", s)
2026-05-30 04:04:11.684666: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
CUDA Graph: support check passed, auto-enabled
GPT_SoVITS/pretrained_models/v2Pro/s2Gv2ProPlus.pth v2 v2ProPlus False
loading sovits_v2ProPlus <All keys matched successfully>
Running on local URL: